In [17]:
from langgraph.graph import StateGraph, START, MessagesState
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
from langgraph.checkpoint.memory import InMemorySaver

In [6]:
load_dotenv()

True

In [8]:
model = ChatGroq(
    model=os.getenv("LLM_MODEL"),
    api_key=os.getenv("LLM_API_KEY")
)

In [9]:
def call_model(state: MessagesState):
    response = model.invoke(state["messages"])
    return {"messages": [response]}

In [18]:
builder = StateGraph(MessagesState)

builder.add_node("call_model", call_model)

builder.add_edge(START, "call_model")

In [19]:
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [20]:
config = {"configurable": {"thread_id":"thread-1"}}

In [21]:
graph.invoke({"messages": [{"role":"user", "content":"Hi! My name is Bhushan."}]}, config=config)

{'messages': [HumanMessage(content='Hi! My name is Bhushan.', additional_kwargs={}, response_metadata={}, id='743f7842-44c9-489d-b7a8-cb15b9900498'),
  AIMessage(content="Hello Bhushan! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 44, 'total_tokens': 72, 'completion_time': 0.040408678, 'completion_tokens_details': None, 'prompt_time': 0.005924083, 'prompt_tokens_details': None, 'queue_time': 0.049329886, 'total_time': 0.046332761}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f8429-d187-74d1-9dbf-fce14e6acb40-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 44, 'output_tokens': 28, 'total_tokens': 72})]}

In [22]:
graph.invoke({"messages":[{"role":"user","content":"What is my name?"}]}, config=config)

{'messages': [HumanMessage(content='Hi! My name is Bhushan.', additional_kwargs={}, response_metadata={}, id='743f7842-44c9-489d-b7a8-cb15b9900498'),
  AIMessage(content="Hello Bhushan! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 44, 'total_tokens': 72, 'completion_time': 0.040408678, 'completion_tokens_details': None, 'prompt_time': 0.005924083, 'prompt_tokens_details': None, 'queue_time': 0.049329886, 'total_time': 0.046332761}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f8429-d187-74d1-9dbf-fce14e6acb40-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 44, 'output_tokens': 28, 'total_tokens': 72}),
  HumanMessage(content='What is my name?', additional_kwargs={}, respo

In [23]:
snap = graph.get_state(config)
vals = snap.values

for m in vals.get("messages", []):
    print("-", type(m).__name__, ":", m.content)

- HumanMessage : Hi! My name is Bhushan.
- AIMessage : Hello Bhushan! It's nice to meet you. Is there something I can help you with or would you like to chat?
- HumanMessage : What is my name?
- AIMessage : Your name is Bhushan. You told me that when you introduced yourself earlier.
